In [4]:
import sys
import os

# Get the absolute path of the directory you want to add
directory_to_add = os.path.abspath("..")

# Add the directory to the beginning of sys.path
sys.path.insert(0, directory_to_add)

# Import the Py2D_solver function from the py2d.Py2D_solver module
from py2d.Py2D_solver import Py2D_solver
from py2d.datamanager import gen_path, get_last_file
from py2d.filter import filter2D_2DFHIT, coarse_spectral_filter_square_2DFHIT, spectral_filter_square_same_size_2DFHIT
from py2d.eddy_viscosity_models import nabla_squared_omega
from py2d.initialize import initialize_wavenumbers_2DFHIT
from py2d.SGSModel import SGSModel
from py2d.convert import Omega2Psi_2DFHIT_spectral, Psi2UV_2DFHIT_spectral

from parameter_recovery_eddy_viscosity import (
    calculate_difference,
    compute_eddy_viscosity_coeff_step,
)
from plot import plot_omega_high_low

# Import the numpy library for numerical operations
import numpy as np
#import os
import shutil

from scipy.io import loadmat, savemat

from matplotlib import pyplot as plt

In [ ]:
# system parameters
Re = 20e3
fkx = 4
fky = 4
alpha = 0.1
beta = 0

# set parameters for high_res simulation
#dt_high_res = 5e-5
dt_high_res = 2.5e-4
#NX_high_res = 1024
NX_high_res = 64


# set parameters for low_res simulation
#dt_low_res = 5*dt_high_res
dt_low_res = dt_high_res
NX_low_res = 64
SGSModel_string = "LEITH"

# Set an initial value for the eddy viscosity coefficient
eddyViscosityCoeff_temp = 0.17

# Set the time interval to update the eddy viscosity coefficient
t_update_eddy_viscosity = 10*dt_low_res

# Nudging (JPW: this was set to 1 before...we want it much higher than that)
mu = 0.5/dt_low_res  # Nudging coefficient

# Parameter updating (JPW: the learning rate for the optimization algorithm)
lr = 1e-6  #  Learning rate

# The high resolution simulation and the low resolution simulation are runnning simultaneously

#JPW: In order to do the updates at the same time for the same resolution we need to step both through a single time step first...
Omega_high_res = Py2D_solver(
        Re=Re,  # Reynolds number
        fkx=fkx,  # Forcing wavenumber in x
        fky=fky,  # Forcing wavenumber in y
        alpha=alpha,  # Rayleigh drag coefficient
        beta=beta,  # Coriolis parameter
        NX=NX_high_res,  # Number of grid points in x and y (options: 32, 64, 128, 256, 512)
        SGSModel_string=SGSModel_string,  # SGS model to use (options: 'NoSGS', 'SMAG', 'DSMAG', 'LEITH', etc.)
        eddyViscosityCoeff=eddyViscosityCoeff_temp,  # Coefficient for eddy viscosity models
        dt=dt_high_res,  # Time step
        saveData=True,  # Flag to save data
        tSAVE=dt_high_res,  # Time interval to save data
        tTotal=dt_low_res,  # Total time of simulation
        readTrue=False, 
        ICnum=1,  # Initial condition number (options: 1 to 20)
        direct_IC=None,
        error_term_hat = 0.0,
        resumeSim=resumeSimulation,  # Flag to start a new simulation or resume an existing one
    )




for count_update_eddy_viscosity_parameter in range(5):

    for count_low_res in range(int(t_update_eddy_viscosity//dt_low_res)):

    ############################# High Res Simulation #############################

        # Running high resolution simulation for length of time = dt_low_res

        print(f"\n************** High Res Simulation **************")

        if count_low_res == 0 and count_update_eddy_viscosity_parameter == 0:
            # Initialize a flag to control whether to resume a simulation
            resumeSimulation = False
        else:
            resumeSimulation = True

        Omega_high_res = Py2D_solver(
                Re=Re,  # Reynolds number
                fkx=fkx,  # Forcing wavenumber in x
                fky=fky,  # Forcing wavenumber in y
                alpha=alpha,  # Rayleigh drag coefficient
                beta=beta,  # Coriolis parameter
                NX=NX_high_res,  # Number of grid points in x and y (options: 32, 64, 128, 256, 512)
                SGSModel_string=SGSModel_string,  # SGS model to use (options: 'NoSGS', 'SMAG', 'DSMAG', 'LEITH', etc.)
                eddyViscosityCoeff=eddyViscosityCoeff_temp,  # Coefficient for eddy viscosity models
                dt=dt_high_res,  # Time step
                saveData=True,  # Flag to save data
                tSAVE=dt_high_res,  # Time interval to save data
                tTotal=dt_low_res,  # Total time of simulation
                readTrue=False, 
                ICnum=1,  # Initial condition number (options: 1 to 20)
                direct_IC=None,
                error_term_hat =0.0,
                resumeSim=resumeSimulation,  # Flag to start a new simulation or resume an existing one
            )

        ## Example code to last 10 snapshots of the Vorticity  (Omega) data from the high resolution simulation

        # Get the path to the directory where the data is saved
        _, SAVE_DIR_DATA_high_res, SAVE_DIR_IC_high_res = gen_path(NX=NX_high_res, dt=dt_high_res, ICnum=1, Re=Re, fkx=fkx, fky=fky, alpha=alpha, beta=beta, SGSModel_string='NoSGS')

        # Individual file contains data for single snapshot with files numbered in the asceding order of saving
        # last_file_number_data is the latest saved file number
        last_file_number_data_high_res = get_last_file(SAVE_DIR_DATA_high_res)

        Omega_high_res_snapshots = []
        for count in range(int(dt_low_res//dt_high_res)):
            print(count)
            print(last_file_number_data_high_res)
            filenumber = last_file_number_data_high_res - count
            filename = f"{SAVE_DIR_DATA_high_res}{filenumber}.mat"

            data = loadmat(filename)

            # Last 10 snapshots are saved in the following array
            Omega_high_res_snapshots.append(data['Omega'])

    ############################# Low Res Simulation #############################

        # Running low resolution simulation for single time step
            
        print(f"\n************** Low Res Simulation **************")

        if count_low_res == 0 and count_update_eddy_viscosity_parameter == 0:
            direct_IC_low_res = None

            # Initialize error term
            error_term_hat = 0
            resumeSimulation_low_res = False
        else:
            resumeSimulation_low_res = True

            # Providing parameters to correct low-res and high-res simulationsapp

        Omega_low_res = Py2D_solver(
                Re=Re,  # Reynolds number
                fkx=fkx,  # Forcing wavenumber in x
                fky=fky,  # Forcing wavenumber in y
                alpha=alpha,  # Rayleigh drag coefficient
                beta=beta,  # Coriolis parameter
                NX=NX_low_res,  # Number of grid points in x and y (options: 32, 64, 128, 256, 512)
                SGSModel_string=SGSModel_string,  # SGS model to use (options: 'NoSGS', 'SMAG', 'DSMAG', 'LEITH', etc.)
                
                # This is the term we update (e.g., C_s or C_l).
                eddyViscosityCoeff=eddyViscosityCoeff_temp*0.9,  # Coefficient for eddy viscosity models initialized to 90% of the actual value
                # Future: Run autodiff on SGS terms.

                dt=dt_low_res,  # Time step
                saveData=True,  # Flag to save data
                tSAVE=dt_low_res,  # Time interval to save data
                tTotal=dt_low_res,  # Total time of simulation
                readTrue=False, 
                ICnum=2,  # Initial condition number (options: 1 to 20)
                direct_IC=direct_IC_low_res,
                error_term_hat = error_term_hat,
                resumeSim=resumeSimulation_low_res,  # Flag to start a new simulation or resume an existing one
            )
        
        ############## Calculating the error term ###################
        # Calculating the error term using the high-res and low-res data
        Omega_high_res_hat = np.fft.fft2(Omega_high_res)
        Omega_low_res_hat = np.fft.fft2(Omega_low_res)

        error_term_hat = calculate_difference(
            Omega_high_res_hat, Omega_low_res_hat, mu
        )

    ############################## Update Eddy Viscosity ##############################
        
    # Calculate the eddy viscosity coefficient using the last 10 snapshots low resolution simulation

    # Get the path to the directory where the data is saved
    _, SAVE_DIR_DATA_low_res, SAVE_DIR_IC_low_res = gen_path(NX=NX_low_res, dt=dt_low_res, ICnum=2, Re=Re, fkx=fkx, fky=fky, alpha=alpha, beta=beta, SGSModel_string='LEITH')


    # Individual file contains data for single snapshot with files numbered in the asceding order of saving
    # last_file_number_data is the latest saved file number
    last_file_number_data_low_res = get_last_file(SAVE_DIR_DATA_low_res)

    Omega_low_res_snapshots = []
    for count in range(int(t_update_eddy_viscosity//dt_low_res)):
        filenumber = last_file_number_data_low_res - count
        filename = f"{SAVE_DIR_DATA_low_res}{filenumber}.mat"

        data = loadmat(filename)

        # Last 10 snapshots are saved in the following array
        Omega_low_res_snapshots.append(data['Omega'])

    # Omega_low_res_snapshots can be used to update the eddy viscosity coefficient

    Omega_high_res_hat = np.fft.fft2(Omega_high_res)
    Omega_low_res_hat = np.fft.fft2(Omega_low_res)

    step = compute_eddy_viscosity_coeff_step(
        SGSModel_string,
        Omega_high_res,
        Omega_low_res,
        NX_low_res,
        eddyViscosityCoeff_temp,
        mu,
        lr,
    )
    eddyViscosityCoeff_temp += step

    print("\n" + "*" * 80)
    print(f"Updated eddy viscosity coefficient = {eddyViscosityCoeff_temp}")

    diff = calculate_difference(Omega_high_res_hat, Omega_low_res_hat, 1)
    abs_freq_error = np.linalg.norm(diff)
    print(f"Error: {abs_freq_error}")
    print("*" * 80 + "\n\n")

plot_omega_high_low(Omega_high_res_hat, Omega_low_res_hat, NX_low_res)
plt.show()

# Remove all results so that this cell can run again immediately.
shutil.rmtree("results")

#     # Just keep the last 10 snapshots of the high resolution simulation Delete the rest
#     for count in range(10):
#         filenumber = last_file_number_data - count - 1
#         filename = f"{SAVE_DIR_DATA_high_res}{filenumber}.mat"
#         os.remove(filename)


************** High Res Simulation **************
Time = 2.50e-04 -- Eddy Turnover Time = 2.57e-01 -- C = 1.70e-01 -- Eddy viscosity = 7.98e-04 ** 
0
None


TypeError: unsupported operand type(s) for -: 'NoneType' and 'int'

In [7]:
for count in range(1):
    print(count)

0
